In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Input

In [3]:
train_df = pd.read_csv("../Dataset/train_metadata.csv")
valid_df = pd.read_csv("../Dataset/valid_metadata.csv")
test_df = pd.read_csv("../Dataset/test_metadata.csv")

In [4]:
train_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,Height,Width
0,HAM_0005972,ISIC_0033319,nv,histo,35.0,female,lower extremity,../Dataset/HAM10000_images_part_2\ISIC_0033319...,450,600
1,HAM_0004902,ISIC_0030823,nv,follow_up,40.0,male,trunk,../Dataset/HAM10000_images_part_2\ISIC_0030823...,450,600
2,HAM_0005282,ISIC_0028730,akiec,histo,65.0,male,lower extremity,../Dataset/HAM10000_images_part_1\ISIC_0028730...,450,600
3,HAM_0000475,ISIC_0027299,nv,follow_up,40.0,male,lower extremity,../Dataset/HAM10000_images_part_1\ISIC_0027299...,450,600
4,HAM_0000949,ISIC_0032444,nv,histo,65.0,male,back,../Dataset/HAM10000_images_part_2\ISIC_0032444...,450,600
...,...,...,...,...,...,...,...,...,...,...
8007,HAM_0000940,ISIC_0032692,vasc,histo,35.0,female,lower extremity,../Dataset/HAM10000_images_part_2\ISIC_0032692...,450,600
8008,HAM_0005629,ISIC_0029317,nv,follow_up,45.0,female,upper extremity,../Dataset/HAM10000_images_part_2\ISIC_0029317...,450,600
8009,HAM_0004025,ISIC_0025983,nv,histo,20.0,female,abdomen,../Dataset/HAM10000_images_part_1\ISIC_0025983...,450,600
8010,HAM_0004542,ISIC_0027256,vasc,consensus,0.0,female,back,../Dataset/HAM10000_images_part_1\ISIC_0027256...,450,600


In [5]:
test_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,Height,Width
0,HAM_0006303,ISIC_0026911,nv,histo,55.0,male,abdomen,../Dataset/HAM10000_images_part_1\ISIC_0026911...,450,600
1,HAM_0002614,ISIC_0027637,nv,follow_up,75.0,male,trunk,../Dataset/HAM10000_images_part_1\ISIC_0027637...,450,600
2,HAM_0004362,ISIC_0031873,nv,follow_up,45.0,female,lower extremity,../Dataset/HAM10000_images_part_2\ISIC_0031873...,450,600
3,HAM_0006207,ISIC_0030034,mel,histo,50.0,female,back,../Dataset/HAM10000_images_part_2\ISIC_0030034...,450,600
4,HAM_0000801,ISIC_0025619,nv,histo,20.0,female,back,../Dataset/HAM10000_images_part_1\ISIC_0025619...,450,600
...,...,...,...,...,...,...,...,...,...,...
997,HAM_0006197,ISIC_0028332,nv,histo,40.0,male,back,../Dataset/HAM10000_images_part_1\ISIC_0028332...,450,600
998,HAM_0006935,ISIC_0026628,nv,follow_up,50.0,male,back,../Dataset/HAM10000_images_part_1\ISIC_0026628...,450,600
999,HAM_0004607,ISIC_0031642,mel,histo,50.0,female,back,../Dataset/HAM10000_images_part_2\ISIC_0031642...,450,600
1000,HAM_0004399,ISIC_0028905,nv,histo,20.0,female,face,../Dataset/HAM10000_images_part_1\ISIC_0028905...,450,600


In [7]:
valid_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,Height,Width
0,HAM_0005031,ISIC_0025109,nv,follow_up,65.0,male,abdomen,../Dataset/HAM10000_images_part_1\ISIC_0025109...,450,600
1,HAM_0006188,ISIC_0024330,df,consensus,40.0,male,lower extremity,../Dataset/HAM10000_images_part_1\ISIC_0024330...,450,600
2,HAM_0007094,ISIC_0026785,nv,follow_up,30.0,female,lower extremity,../Dataset/HAM10000_images_part_1\ISIC_0026785...,450,600
3,HAM_0006129,ISIC_0029560,nv,histo,45.0,male,lower extremity,../Dataset/HAM10000_images_part_2\ISIC_0029560...,450,600
4,HAM_0005351,ISIC_0032590,nv,histo,20.0,female,lower extremity,../Dataset/HAM10000_images_part_2\ISIC_0032590...,450,600
...,...,...,...,...,...,...,...,...,...,...
996,HAM_0001187,ISIC_0027914,nv,follow_up,45.0,male,trunk,../Dataset/HAM10000_images_part_1\ISIC_0027914...,450,600
997,HAM_0006973,ISIC_0029377,nv,follow_up,60.0,male,trunk,../Dataset/HAM10000_images_part_2\ISIC_0029377...,450,600
998,HAM_0006723,ISIC_0026365,bkl,histo,45.0,male,lower extremity,../Dataset/HAM10000_images_part_1\ISIC_0026365...,450,600
999,HAM_0000434,ISIC_0031635,nv,follow_up,45.0,female,trunk,../Dataset/HAM10000_images_part_2\ISIC_0031635...,450,600


# Encode Label

In [8]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])


NUM_CLASSES = len(label_encoder.classes_)

## Compute Class Weights

In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

## Create Image Loader

In [10]:
IMG_SIZE = (224,224)

def process_image(path,label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image,channels=3)

    image = tf.image.resize(image,IMG_SIZE)

    image = tf.cast(image,tf.float32)/255.0

    return image,label

## Create TensorFlow Dataset

In [11]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"],train_df["label"])
)

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"],valid_df["label"])
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df["path"],test_df["label"])
)

train_dataset = train_dataset.map(process_image)
valid_dataset = valid_dataset.map(process_image)
test_dataset = test_dataset.map(process_image)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
valid_dataset = valid_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)